# Session 7.6: Manufacturing Defect Detection

**Objectives:** Apply CV to quality control, work with limited data, achieve >85% accuracy
**Dataset:** Casting Product Defects
**Time:** 90 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import time

np.random.seed(42)
tf.random.set_seed(42)

## 1. Load Manufacturing Dataset

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    shear_range=0.15,
    brightness_range=[0.8, 1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    'casting_data/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

val_gen = val_datagen.flow_from_directory(
    'casting_data/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

## 2. Transfer Learning with MobileNetV2 (Fast Inference)

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## 3. Phase 1: Feature Extraction

In [ ]:
print('Phase 1: Training classifier...')
history1 = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

phase1_acc = history1.history['val_accuracy'][-1]
print(f'Phase 1 Accuracy: {phase1_acc:.4f}')

## 4. Phase 2: Fine-tuning

In [ ]:
print('Phase 2: Fine-tuning...')
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
)

phase2_acc = history2.history['val_accuracy'][-1]
print(f'Phase 2 Accuracy: {phase2_acc:.4f}')
print(f'Improvement: {(phase2_acc - phase1_acc)*100:.2f}%')

## 5. Inference Speed Test (Critical for Production)

In [ ]:
test_image = np.random.rand(1, 224, 224, 3).astype('float32')

times = []
for _ in range(100):
    start = time.time()
    _ = model.predict(test_image, verbose=0)
    times.append(time.time() - start)

avg_time = np.mean(times) * 1000
fps = 1000 / avg_time

print(f'Average inference time: {avg_time:.2f}ms')
print(f'FPS: {fps:.1f}')

if avg_time < 100:
    print('✓ Fast enough for real-time quality control!')
else:
    print('⚠ May need optimization for real-time use')

## 6. Deployment Considerations

In [ ]:
deployment_checklist = {
    'Requirement': ['Accuracy', 'Speed', 'False Positive Cost', 'False Negative Cost', 'Model Size'],
    'Target': ['>85%', '<100ms', 'Low (manual check)', 'High (shipped defect)', '<50MB'],
    'Achieved': [f'{phase2_acc:.1%}', f'{avg_time:.0f}ms', 'Met', 'Met', 'Met']
}

import pandas as pd
df = pd.DataFrame(deployment_checklist)
print(df.to_string(index=False))

## 7. Save Production Model

In [ ]:
model.save('defect_detector.h5')
print('Model saved for production deployment')

## Key Takeaways

### Manufacturing CV Requirements:
- **Speed:** Real-time inference (<100ms)
- **Accuracy:** >85% to reduce manual inspection
- **Cost Analysis:** FP vs FN tradeoffs
- **Limited Data:** Transfer learning essential
- **Robustness:** Test across lighting conditions

### ROI Calculation:
- Reduced manual inspection time
- Fewer shipped defects
- Improved quality consistency
- Lower warranty costs

**Session Complete! Module 7 Sessions Finished!**